# 📈 IDX Bandarmology — End-to-End Notebook

This notebook runs the full flow: **scrape → clean → pipeline → analysis → modeling**,
to test the hypothesis:

> *Do smart-money accumulation signals in IDX stocks align with stronger future price moves?*

**Data sources:**
- `yfinance` — free historical OHLCV prices.
- Private broker-flow endpoint — broker summary, accumulation/distribution signals, foreign/domestic flow. Requires `BROKER_API_TOKEN` in `.env` (see `.env.example`).

**How to use:** run the cells from top to bottom. Edit `WATCHLIST` in the next cell to change the tickers.
Run this notebook on each trading day (or schedule it) to build more history — more days make the analysis more reliable.

## 1. Setup

In [ ]:
import importlib
import sys
from pathlib import Path

# repo root is the parent of this notebooks/ folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
importlib.invalidate_caches()
for name in [n for n in list(sys.modules) if n == "idx_bandarmology" or n.startswith("idx_bandarmology.")]:
    del sys.modules[name]

import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
%matplotlib inline
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from idx_bandarmology import broker_api, config, pipeline, storage, features, analysis, modeling

print("Broker API token configured:", broker_api.is_available())
print("Default watchlist:", config.WATCHLIST)

## 2. Watchlist

Edit the ticker list below. You do not need the `.JK` suffix — it is added automatically for yfinance
and removed automatically for the broker-flow client.

In [ ]:
WATCHLIST = ["BBCA", "BBRI", "BMRI", "BBNI", "TLKM", "ASII", "UNVR", "GOTO", "BREN", "ANTM"]

# Or inspect a new ticker before adding it to the watchlist:
# from idx_bandarmology import broker_api
# broker_api.fetch_analysis("ADRO")["summary"]

## 3. Scrape data and run the pipeline

This single function does everything:
1. Fetch historical prices from yfinance for every ticker in the watchlist.
2. Fetch the latest broker-flow snapshot for each ticker (if a token is available).
3. Save everything to SQLite (`data/db/bandarmology.sqlite`), which is also used by the Streamlit dashboard.

Run this cell on each trading day to build history. The broker-flow endpoint provides a latest snapshot,
not a historical range, so the time series grows as often as you run the pipeline.

In [ ]:
import importlib
import idx_bandarmology.config
import idx_bandarmology.broker_api
import idx_bandarmology.pipeline

importlib.reload(idx_bandarmology.config)
importlib.reload(idx_bandarmology.broker_api)
importlib.reload(idx_bandarmology.pipeline)

from idx_bandarmology import broker_api, pipeline
print("Broker API token configured:", broker_api.is_available())

result = pipeline.run(WATCHLIST, price_period="2y")

print(f"Prices: {result['n_prices']} rows")
print(f"Broker flow: {result['n_broker']} rows")

### 3a. Inspect raw scrape results per ticker (optional)

Useful for checking raw broker-flow data before it lands in the pipeline. Each section (`broker`, `foreignDomestic`,
`pricePerformance`) is independent, so one failure does not hide the others.

In [ ]:
for sym, r in result["broker_results"].items():
    print("=" * 70)
    print(sym, "-", r.get("summary"))
    if r.get("broker", {}).get("available"):
        print(" ", r["broker"]["conclusion"])
    if r.get("foreignDomestic", {}).get("available"):
        print(" ", r["foreignDomestic"]["conclusion"])

## 4. Inspect raw pipeline tables with pandas

The main SQLite tables are:
- `prices` — daily OHLCV by ticker.
- `broker_flow` — one broker-flow snapshot per ticker per pipeline run date.

In [ ]:
price_df = storage.read_prices(WATCHLIST)
broker_df = storage.read_broker_flow(WATCHLIST)

print("prices:", price_df.shape)
display(price_df.tail())

print("\nbroker_flow:", broker_df.shape)
display(broker_df.tail())

## 5. Feature engineering

`features.build_feature_table()` merges `prices` and `broker_flow` into one tidy table,
and computes **backward returns** so the analysis is populated immediately even if broker snapshots only exist for one day.
Forward returns are still stored too, but they only become usable after more trading days accumulate.

In [ ]:
feat = features.build_feature_table(WATCHLIST, horizons=(1, 3, 5, 10))
print(feat.shape)
feat.tail(10)[["ticker", "date", "close", "bandar_signal", "foreign_net_flow_pct",
               "back_return_1d", "back_return_5d", "back_return_10d"]]

## 6. Descriptive analysis and correlations

Core question: when today's smart-money signal shows **accumulation**, does the stock tend to behave better
than when the signal is **distribution** or neutral?

In [ ]:
corr = analysis.correlation_table(feat)
display(corr)

In [ ]:
summary = analysis.summary_by_signal(feat, target_col="back_return_5d")
display(summary)

In [ ]:
fig = analysis.plot_signal_bucket_returns(feat, target_col="back_return_5d")
plt.show()

In [ ]:
fig = analysis.plot_signal_vs_return(feat, horizon=5, direction="back")
plt.show()

In [ ]:
# Per-ticker correlation — the effect can differ by stock
analysis.correlation_by_ticker(feat, target_col="back_return_5d")

In [ ]:
# View price and smart-money signal for one ticker
fig = analysis.plot_price_with_signal(feat, WATCHLIST[0])
plt.show()

## 7. Modeling and hypothesis testing

Two approaches:

1. **Linear regression (OLS)** — is there a statistically significant relationship (p < 0.05)
   between smart-money signals and historical returns while controlling for other variables?
2. **Classification (Logistic Regression / Random Forest)** — if we convert the question to
   a binary up/down target, how accurately do the signals classify it?

In [ ]:
reg = modeling.linear_regression(feat, target_col="back_return_5d")
print(f"n = {reg.n_obs}, R-squared = {reg.r_squared:.4f}")
display(reg.coefficients)

In [ ]:
# Full statsmodels summary (optional, more detail)
print(reg.summary_text)

In [ ]:
clf_logit = modeling.classify_up_down(feat, target_col="back_return_5d", model_type="logistic")
print(f"Logistic Regression — n={clf_logit.n_obs}, accuracy={clf_logit.accuracy:.1%}, "
      f"precision={clf_logit.precision:.1%}, recall={clf_logit.recall:.1%}, "
      f"ROC-AUC={clf_logit.roc_auc}")
display(clf_logit.feature_importance)

In [ ]:
clf_rf = modeling.classify_up_down(feat, target_col="back_return_5d", model_type="random_forest")
print(f"Random Forest — n={clf_rf.n_obs}, accuracy={clf_rf.accuracy:.1%}, "
      f"precision={clf_rf.precision:.1%}, recall={clf_rf.recall:.1%}, "
      f"ROC-AUC={clf_rf.roc_auc}")
display(clf_rf.feature_importance)

## 8. Conclusion

In [ ]:
print(modeling.hypothesis_verdict(reg, clf_logit))

## 9. Dashboard

For interactive exploration (filter tickers, view all charts together, rerun the pipeline from the UI), open the Streamlit dashboard:

```bash
streamlit run dashboard/app.py
```

The dashboard reads from the same SQLite database as this notebook, so both stay in sync. Run the pipeline
from this notebook or from the dashboard sidebar, then refresh.

> **Publishing tip:** screenshot the "Modeling / Forecasts" tab in the dashboard, or export one of the charts above
> with `fig.savefig("chart.png", dpi=150)` for a repo cover image.